# ACE-Step 1.5: three score candidates for The Heritage Short

Generates 3 instrumental candidates (A, B, C) from `storyboard.json`'s music brief, each ~72 s, and writes a log you paste into `runs.jsonl`.

**Before you run:** Settings -> Accelerator: *GPU T4 x2* (one GPU is used), Internet: *On*.
Written from the ACE-Step 1.5 docs (`docs/en/INFERENCE.md`, `INSTALL.md`). Anything the docs did not cover (T4 flags, Kaggle) is untested: if a cell fails, copy the error into FAILURES.md.
Model licence: MIT (model card). Weights download from Hugging Face on first run (~10 GB).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python --version

## 1. Install (uv builds an isolated environment, so the notebook's own Python is untouched)

In [ ]:
import time
t_install = time.time()
!pip -q install uv
!git clone -q https://github.com/ACE-Step/ACE-Step-1.5.git /kaggle/working/ACE-Step-1.5
%cd /kaggle/working/ACE-Step-1.5
!uv sync
print('install seconds:', round(time.time() - t_install))

## 2. Download weights (turbo DiT + the small 0.6B language model)

In [ ]:
t_dl = time.time()
!uv run acestep-download
!uv run huggingface-cli download ACE-Step/acestep-5Hz-lm-0.6B --local-dir ./checkpoints/acestep-5Hz-lm-0.6B
print('download seconds:', round(time.time() - t_dl))

## 3. The generation script
Edit the three captions if you like. Same duration for all so they are comparable. `shift=3.0` is the documented setting for the turbo model.

In [ ]:
%%writefile /kaggle/working/gen.py
import json, os, shutil, time, traceback
from pathlib import Path
from acestep.handler import AceStepHandler
from acestep.llm_inference import LLMHandler
from acestep.inference import GenerationParams, GenerationConfig, generate_music

ROOT = os.getcwd()
OUT = Path('/kaggle/working/music'); OUT.mkdir(parents=True, exist_ok=True)
LOG = Path('/kaggle/working/kaggle_runs.jsonl')
DURATION = 72  # film body 64 s + 4 s end card, plus a little to trim
BASE = 'instrumental, no vocals, warm and hopeful, slow tempo, '
CANDIDATES = {
    'A': (BASE + 'soft plucked acoustic guitar and light hand percussion, Ethiopian pentatonic melody', 101),
    'B': (BASE + 'gentle piano with a warm string pad, simple repeating motif, patient and calm', 202),
    'C': (BASE + 'kalimba and marimba with a soft drone, sparse and intimate, like a lullaby', 303),
}
MODEL, LM = 'acestep-v15-turbo', 'acestep-5Hz-lm-0.6B'

def log(rec):
    with open(LOG, 'a') as f:
        f.write(json.dumps(rec) + '\n')

t = time.time()
dit = AceStepHandler()
dit.initialize_service(project_root=ROOT, config_path=MODEL, device='cuda')
llm = LLMHandler()
llm.initialize(checkpoint_dir=os.path.join(ROOT, 'checkpoints'), lm_model_path=LM, backend='pt', device='cuda')
load_s = time.time() - t
print('models loaded in', round(load_s), 's')

for name, (caption, seed) in CANDIDATES.items():
    params = GenerationParams(caption=caption, lyrics='[Instrumental]', instrumental=True,
                              duration=DURATION, seed=seed, inference_steps=8, shift=3.0)
    config = GenerationConfig(batch_size=1, audio_format='wav', use_random_seed=False, seeds=[seed])
    t = time.time()
    try:
        result = generate_music(dit, llm, params, config, save_dir=str(OUT / f'raw_{name}'))
        ok, err = bool(result.success), getattr(result, 'error', None)
    except Exception:
        ok, err, result = False, traceback.format_exc(), None
    dt = time.time() - t
    rec = {'stage': 'music', 'model': f'ACE-Step-1.5 {MODEL} + {LM}', 'candidate': name, 'seed': seed,
           'params': {'caption': caption, 'duration': DURATION, 'steps': 8, 'shift': 3.0},
           'latency_s': round(dt, 1), 'gpu': 'Kaggle T4', 'gpu_minutes': round(dt / 60, 2),
           'model_load_s': round(load_s, 1), 'success': ok, 'cost_usd': 0.0}
    if ok:
        shutil.copy(result.audios[0]['path'], OUT / f'cand_{name}.wav')
        print(name, 'ok in', round(dt), 's')
    else:
        rec['error'] = str(err)[:500]
        print(name, 'FAILED:', str(err)[:500])
    log(rec)


## 4. Run it (from inside the repo so `uv run` uses the right environment)

In [ ]:
%cd /kaggle/working/ACE-Step-1.5
!uv run python /kaggle/working/gen.py

## 5. Listen, then download
Download `/kaggle/working/music/cand_A.wav`, `cand_B.wav`, `cand_C.wav` and `/kaggle/working/kaggle_runs.jsonl` (Output tab on the right after *Save Version*, or the file browser). Put the wavs in `outputs/music/` in the repo and append the log lines to `runs.jsonl`.

In [ ]:
from IPython.display import Audio, display
import os
for n in 'ABC':
    p = f'/kaggle/working/music/cand_{n}.wav'
    if os.path.exists(p):
        print('Candidate', n); display(Audio(p))
print(open('/kaggle/working/kaggle_runs.jsonl').read())